# Load data from the CSV

Using data from https://s3.amazonaws.com/tripdata/index.html. Put any csvs in citibike-tripdata/\<csvs here\>

In [1]:
import pandas as pd

file_names = [
    "citibike-tripdata/202609-citibike-tripdata_1.csv",
    "citibike-tripdata/202609-citibike-tripdata_2.csv",
    "citibike-tripdata/202609-citibike-tripdata_3.csv",
]

dfs = [pd.read_csv(file_name) for file_name in file_names]

df = pd.concat(dfs, ignore_index=True)

/var/folders/4j/39rtwmqd5tg4g8dn1t6m037h0000gn/T/ipykernel_61400/2459551289.py:9: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(file_name) for file_name in file_names]
/var/folders/4j/39rtwmqd5tg4g8dn1t6m037h0000gn/T/ipykernel_61400/2459551289.py:9: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(file_name) for file_name in file_names]
/var/folders/4j/39rtwmqd5tg4g8dn1t6m037h0000gn/T/ipykernel_61400/2459551289.py:9: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(file_name) for file_name in file_names]


# Looking at the data

In [2]:
station_name_value_pairs = df['start_station_name'].value_counts()
print(station_name_value_pairs)

start_station_name
W 21 St & 6 Ave                 10092
Pier 61 at Chelsea Piers         9519
W 31 St & 7 Ave                  8866
E 17 St & Broadway               8804
9 Ave & W 33 St                  8579
                                ...  
Willmohr St & Rockaway Pkwy         2
Riverdale Ave & New Lots Ave        1
Stratford Rd & Beverley Rd          1
Sheffield Ave & Linden Blvd         1
Arlington Ave & Warwick St          1
Name: count, Length: 2371, dtype: int64


In [3]:
top_pairs = df[['start_station_name', 'end_station_name']].value_counts().head(60)
print(top_pairs)

start_station_name                 end_station_name                 
Central Park S & 6 Ave             Central Park S & 6 Ave               1601
7 Ave & Central Park South         7 Ave & Central Park South           1264
Central Park W & W 72 St           Central Park W & W 72 St              645
Roosevelt Island Tramway           Roosevelt Island Tramway              603
Soissons Landing                   Soissons Landing                      579
55 Ave & Center Blvd               Vernon Blvd & 50 Ave                  492
Central Park W & W 68 St           Central Park W & W 68 St              485
Franklin St & Dupont St            Jackson Ave & 50 Ave                  481
Central Park S & Grand Army Plaza  Central Park S & Grand Army Plaza     453
Vernon Blvd & 50 Ave               55 Ave & Center Blvd                  448
Eastern Pkwy & Kingston Ave        Eastern Pkwy & Kingston Ave           433
Eagle St & Manhattan Ave           Jackson Ave & 50 Ave                  421
S 4 St 

# Making a station pair dataframe

In [4]:
# ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual

df.drop(columns=["ride_id", "rideable_type", "start_station_id", "end_station_id", "start_lat", "start_lng", "end_lat", "end_lng", "member_casual"], inplace=True)

In [5]:
df_55_vernon = df[(df['start_station_name'] == '55 Ave & Center Blvd') & (df['end_station_name'] == "Vernon Blvd & 50 Ave")].copy()
df_vernon_55 = df[(df['start_station_name'] == 'Vernon Blvd & 50 Ave') & (df['end_station_name'] == "55 Ave & Center Blvd")].copy()

In [6]:
print(df_55_vernon.shape)
print(df_55_vernon.head())

print(df_vernon_55.shape)
print(df_vernon_55.head())

print(df[(df['start_station_name'] == '55 Ave & Center Blvd')].shape)
print(df[(df['start_station_name'] == 'Vernon Blvd & 50 Ave')].shape)
print(df[(df['start_station_name'] == 'Vernon Blvd & 50 Ave') & (df['end_station_name'] == "55 Ave & Center Blvd")].shape)

(492, 4)
                    started_at                 ended_at    start_station_name  \
62130  2026-09-03 12:43:48.177  2026-09-03 12:47:50.123  55 Ave & Center Blvd   
62131  2026-09-10 20:05:47.226  2026-09-10 20:08:47.595  55 Ave & Center Blvd   
62132  2026-09-03 07:25:33.188  2026-09-03 07:30:18.503  55 Ave & Center Blvd   
62133  2026-09-12 09:25:14.958  2026-09-12 09:27:51.131  55 Ave & Center Blvd   
62135  2026-09-05 18:50:57.224  2026-09-05 18:54:15.541  55 Ave & Center Blvd   

           end_station_name  
62130  Vernon Blvd & 50 Ave  
62131  Vernon Blvd & 50 Ave  
62132  Vernon Blvd & 50 Ave  
62133  Vernon Blvd & 50 Ave  
62135  Vernon Blvd & 50 Ave  
(448, 4)
                      started_at                 ended_at  \
1526015  2026-09-09 17:47:52.859  2026-09-09 17:52:01.399   
1526016  2026-09-01 15:11:24.470  2026-09-01 15:18:09.075   
1526017  2026-09-13 17:07:55.505  2026-09-13 17:10:42.754   
1526018  2026-09-02 17:40:51.736  2026-09-02 17:45:05.701   
1526019  2

In [7]:
df_55_vernon["started_at"] = pd.to_datetime(df_55_vernon["started_at"])
df_55_vernon["ended_at"] = pd.to_datetime(df_55_vernon["ended_at"])

df_vernon_55["started_at"] = pd.to_datetime(df_vernon_55["started_at"])
df_vernon_55["ended_at"] = pd.to_datetime(df_vernon_55["ended_at"])


In [8]:
df_55_vernon["start_hour"] = df_55_vernon["started_at"].dt.hour
df_55_vernon["end_hour"] = df_55_vernon["ended_at"].dt.hour

df_vernon_55["start_hour"] = df_vernon_55["started_at"].dt.hour
df_vernon_55["end_hour"] = df_vernon_55["ended_at"].dt.hour

In [9]:
print(df_55_vernon.head())

                   started_at                ended_at    start_station_name  \
62130 2026-09-03 12:43:48.177 2026-09-03 12:47:50.123  55 Ave & Center Blvd   
62131 2026-09-10 20:05:47.226 2026-09-10 20:08:47.595  55 Ave & Center Blvd   
62132 2026-09-03 07:25:33.188 2026-09-03 07:30:18.503  55 Ave & Center Blvd   
62133 2026-09-12 09:25:14.958 2026-09-12 09:27:51.131  55 Ave & Center Blvd   
62135 2026-09-05 18:50:57.224 2026-09-05 18:54:15.541  55 Ave & Center Blvd   

           end_station_name  start_hour  end_hour  
62130  Vernon Blvd & 50 Ave          12        12  
62131  Vernon Blvd & 50 Ave          20        20  
62132  Vernon Blvd & 50 Ave           7         7  
62133  Vernon Blvd & 50 Ave           9         9  
62135  Vernon Blvd & 50 Ave          18        18  


In [10]:
first_df_counts = df_55_vernon['start_hour'].value_counts()

In [11]:
second_df_counts = df_vernon_55['start_hour'].value_counts()

In [12]:
import numpy as np
array_of_observations = np.zeros((24,2))

for i in range(len(array_of_observations)):
    array_of_observations[i,0] = first_df_counts.get(i, 0)
    array_of_observations[i,1] = second_df_counts.get(i, 0)
array_of_observations = array_of_observations / 30 # to get per day
print(array_of_observations)

[[0.06666667 0.1       ]
 [0.         0.        ]
 [0.         0.        ]
 [0.         0.        ]
 [0.03333333 0.03333333]
 [0.53333333 0.        ]
 [2.16666667 0.        ]
 [1.86666667 0.23333333]
 [0.53333333 0.56666667]
 [0.46666667 0.6       ]
 [0.63333333 0.4       ]
 [0.56666667 0.53333333]
 [0.8        0.73333333]
 [0.6        0.53333333]
 [0.66666667 1.5       ]
 [0.66666667 1.2       ]
 [0.96666667 1.96666667]
 [1.53333333 2.06666667]
 [1.56666667 1.43333333]
 [1.06666667 1.2       ]
 [0.76666667 0.63333333]
 [0.43333333 0.43333333]
 [0.26666667 0.46666667]
 [0.2        0.3       ]]


# Create the HMM

In [22]:
from hmmlearn import hmm

model = hmm.GaussianHMM(
    n_components=3, covariance_type="full", n_iter=100, random_state=88888888
)

In [23]:
model.fit(array_of_observations, 24)

GaussianHMM(covariance_type='full', n_components=3, n_iter=100,
            random_state=88888888)

In [24]:
hidden_states = model.predict(array_of_observations, 24)
print(hidden_states)

[1 1 1 1 1 0 2 2 1 0 0 0 0 0 0 0 0 2 2 1 0 0 0 0]
